# Copenhagen OSM vs official accessibility

This notebook is the runnable version of the project pipeline. I kept the real code in `src/` because it is easier to test, reuse, and rerun from the command line. The notebook gives the same workflow in smaller chunks, with notes about what each step is doing.

Run the cells in order for a fresh build. If the processed files already exist, you can jump to the later diagnostic and mapping sections.

## setup

The notebook can be opened from the project root or from the `notebooks/` folder. This cell moves to the project root and defines one small helper for running a stage.

In [ ]:
from pathlib import Path
import os
import runpy

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

SRC_DIR = PROJECT_ROOT / "src"


def run_stage(script_name: str) -> None:
    script_path = SRC_DIR / script_name
    print(f"Running {script_name}")
    runpy.run_path(str(script_path), run_name="__main__")
    print(f"Finished {script_name}")

## quick file check

The OSM PBF is deliberately local. This check fails early if the file is missing, which is better than discovering the problem halfway through the run.

In [ ]:
from pathlib import Path

pbf_path = PROJECT_ROOT / "data" / "raw" / "osm" / "denmark-latest.osm.pbf"
if not pbf_path.exists():
    raise FileNotFoundError(
        f"Place the Denmark OSM PBF at {pbf_path}. The project does not download it."
    )
print(f"Found local PBF: {pbf_path}")

## Download the city data

This pulls the four Copenhagen datasets from Open Data DK and keeps the CKAN metadata. I like saving the source links here because open-data portals change more often than anyone wants.

In [ ]:
run_stage("01_download_official_data.py")

## Clean the official layers

The official files come in with their own field names and geometry quirks. This step inspects them, clips them to Copenhagen, and writes a common amenity schema.

In [ ]:
run_stage("03_clean_official_amenities.py")

## Extract OSM from the local PBF

This uses the local Denmark PBF only. No Overpass, no Geofabrik download. The same OSM walking network will be used for both accessibility scenarios.

In [ ]:
run_stage("02_extract_osm_data.py")

## Clean the OSM amenities

OSM points and polygons get converted into the same schema as the official amenities. For routing, polygons become representative points, while the original geometries stay on disk.

In [ ]:
run_stage("04_clean_osm_amenities.py")

## Prepare the walking network

The walking edges are projected to EPSG:25832, lengths are measured in metres, and walking time is calculated at 5 km/h.

In [ ]:
run_stage("05_prepare_walking_network.py")

## Build the 500 m origin grid

The grid is intentionally simple. It gives us a regular set of origins, but the edge cells need care later because Copenhagen has water, islands, and awkward boundaries.

In [ ]:
run_stage("06_create_origin_grid.py")

## Match OSM and official POIs

Before routing, this checks how many OSM amenities match an official amenity nearby. It is a useful sanity check on completeness and classification.

In [ ]:
run_stage("07_match_osm_to_official.py")

## Compute baseline accessibility

For each source and amenity type, this uses multi-source Dijkstra to get the walking time from every origin to the nearest amenity.

In [ ]:
run_stage("08_compute_accessibility.py")

## Classify the baseline differences

This labels each origin as agreement, OSM false access, or OSM hidden access. Those labels make the maps easier to read than raw time differences alone.

In [ ]:
run_stage("09_compare_accessibility.py")

## Make the baseline maps

These are the first-pass maps. They are still useful for comparison, even though the origin-quality checks below produce the recommended final maps.

In [ ]:
run_stage("10_make_maps.py")

## Diagnose origin quality

The baseline run left too many origins unassigned to districts. This script reconstructs that problem and checks which origins sit on boundary or harbour-like edge cells.

In [ ]:
run_stage("11_diagnose_origin_quality.py")

## Assign districts by overlap

Centroid assignment is brittle for coastal grid cells. This assigns each grid cell to the Bydele polygon with the largest area overlap.

In [ ]:
run_stage("12_assign_districts_by_overlap.py")

## Check snapping distances

Snapping hundreds of metres away from a grid centroid can bend the accessibility result. This step flags those origins and makes a histogram plus an outlier map.

In [ ]:
run_stage("13_diagnose_snapping_outliers.py")

## Create full and cleaned origin sets

The baseline stays intact. The clean100 and clean250 sets remove origins with large snapping distances and very weak district overlap.

In [ ]:
run_stage("14_create_clean_origin_set.py")

## Rerun accessibility on the cleaned origins

The walking network and amenity layers stay fixed here. Only the origins change, which keeps the sensitivity test clean.

In [ ]:
run_stage("15_rerun_accessibility_for_clean_origins.py")

## Rebuild district summaries

District summaries now use the largest-overlap assignment. Any origin with no real district overlap gets reported separately instead of being hidden in an Unassigned bucket.

In [ ]:
run_stage("16_rerun_district_summaries.py")

## Compare baseline and cleaned results

This is the robustness check. If the same story holds after cleaning, the findings are much easier to defend.

In [ ]:
run_stage("17_compare_baseline_vs_cleaned.py")

## Make the cleaned maps

The clean100 maps are the ones I would use in the write-up, with the baseline maps kept as a check.

In [ ]:
run_stage("18_make_cleaned_maps.py")

## a few tables worth checking

These tables are the quickest way to see whether the quality-control pass did what it was supposed to do.

In [ ]:
import pandas as pd

tables = [
    "outputs/tables/origin_quality_diagnostic_counts.csv",
    "outputs/tables/origin_cleaning_summary.csv",
    "outputs/tables/robustness_summary_by_origin_set.csv",
]

for table in tables:
    print(f"\n{table}")
    display(pd.read_csv(PROJECT_ROOT / table))

## final note

For reporting, I would use the `clean100` maps and the corrected district summaries. The baseline outputs should stay in the project, though, because they show exactly how much the origin-quality choices mattered.